# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
**https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json**

This dataset contains survey responses from residents of Kilifi County, Kenya, focused on mental health indicators, demographic characteristics, and assessment scores (GAD-7, PHQ-9, PSQ).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}\n")
print(f"Published: {metadata.datePublished}\n")
print(f"Keywords: {', '.join(metadata.keywords)}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Below, we discover what record sets and fields are provided in the dataset and print their IDs so we can extract data accurately.

In [ ]:
# Review record sets in the dataset
record_sets = dataset.record_sets
print('Record sets available:')
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")

# For demonstration, print the first few records from the main record set
main_record_set_id = record_sets[0].id  # Use the first record set
print(f"\nSample records from `{main_record_set_id}`:")
sample_records = list(dataset.records(record_set=main_record_set_id))[:3]
for i, rec in enumerate(sample_records):
    print(f"Record {i+1}: {rec}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We extract all main record sets and display their columns.

In [ ]:
# Extract data from all record sets
dataframes = {}

# Store ids for each record set
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set {record_set_id} columns: {df.columns.tolist()}")

main_rs = record_set_ids[0]
print(f"\nPreview of the main record set ({main_rs}):")
display(dataframes[main_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Below, we filter and analyze numeric fields such as psychological assessment scores, referenced by their specific field `@id`s.

In [ ]:
# Identify a numeric field (e.g., GAD-7 score)
main_rs_fields = record_sets[0].fields

# Find GAD-7 score field
numeric_field_name = 'gad_7_total_score' # Adjust as per your dataset field
numeric_field_id = None
for field in main_rs_fields:
    if field.name.lower() in ['gad_7_total_score', 'gad7_total_score', 'gad_score', 'gad_7_score']:
        numeric_field_id = field.id
        print(f"Numeric field found: {field.name}, @id: {numeric_field_id}")
        break

# Fall back for educational purposes
if numeric_field_id is None:
    numeric_field_id = main_rs_fields[0].id
    print(f"No GAD-7 score found; using first field: {main_rs_fields[0].name}, @id: {numeric_field_id}")

# Filtering
df_main = dataframes[main_rs]

if numeric_field_id in df_main.columns:
    threshold = 10
    filtered_df = df_main[df_main[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head())

    # Grouping by demographic field (e.g., gender)
    group_field_name = 'gender'
    group_field_id = None
    for field in main_rs_fields:
        if field.name.lower() in ['gender', 'sex']:
            group_field_id = field.id
            print(f"Group field found: {field.name}, @id: {group_field_id}")
            break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df)
else:
    print(f"Numeric field {numeric_field_id} not found in DataFrame columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of GAD-7 scores
if numeric_field_id in df_main.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df_main[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (GAD-7 Score)")
    plt.xlabel("GAD-7 Score")
    plt.ylabel("Frequency")
    plt.show()

# Boxplot by gender
if group_field_id and numeric_field_id in df_main.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=df_main[group_field_id], y=df_main[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel("Gender")
    plt.ylabel("GAD-7 Score")
    plt.show()

## 6. Conclusion
The FAIR² Mental Health Survey dataset provides valuable insights into mental health indicators and demographic characteristics among residents of Kilifi County, Kenya.

- The dataset utilizes the Croissant schema for rich metadata and reproducible structure.
- Data exploration reveals distribution of assessment scores (such as GAD-7) and relationships to demographic fields.
- Group-level differences (e.g., by gender) can be quickly visualized and analyzed thanks to the standardized record set and field `@id`s.
- Further research and public health interventions can use this dataset as a foundation for analysis, model development, and community impact.